<a href="https://colab.research.google.com/github/shahdhesham/Thesis_Set1/blob/main/LLAMA_Set1_ZeroShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
from huggingface_hub import login
import os

# 1. Read token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Store it into environment variable (optional but helpful)
os.environ["HF_TOKEN"] = hf_token

# 3. Login to HuggingFace Hub
login(token=hf_token)



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
!hf auth whoami


user:  ShahdH


In [3]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [4]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA NOT available. Using CPU.")

CUDA is available! Using GPU.
GPU device name: NVIDIA A100-SXM4-40GB


In [5]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       1.7Gi        77Gi       1.0Mi       4.1Gi        80Gi
Swap:             0B          0B          0B


In [6]:
from google.colab import files
import zipfile
import torch

import os
from transformers import AutoModelForCausalLM, AutoTokenizer

In [7]:
import shutil
import os

# Delete EVERYTHING (folders AND old zips)
!rm -rf input_folder output_folder *.zip

print("✅ All cleaned up!")
print("\n📂 Current directory:")
!ls -la

✅ All cleaned up!

📂 Current directory:
total 16
drwxr-xr-x 1 root root 4096 Nov 12 14:30 .
drwxr-xr-x 1 root root 4096 Nov 15 23:10 ..
drwxr-xr-x 4 root root 4096 Nov 12 14:30 .config
drwxr-xr-x 1 root root 4096 Nov 12 14:30 sample_data


In [8]:
# 1. Upload ZIP file
print("Upload your ZIP file containing .c files:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# 2. Extract ZIP
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('input_folder')
print("Files extracted to 'input_folder/'")

Upload your ZIP file containing .c files:


Saving C.zip to C.zip
Files extracted to 'input_folder/'


In [9]:
# 3. Load model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer.pad_token = tokenizer.eos_token  # Add this line
tokenizer.padding_side = "left"  # ✅ ADD THIS LINE!



config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [10]:
def translate_batch(c_code_list):
    """
    Translate batch using tokenizer's built-in LEFT padding
    """
    # Build messages
    all_messages = []
    for c_code in c_code_list:
        messages = [
            {"role": "system", "content": "You are an expert code translator..."},
            {"role": "user", "content": f"Translate this C code to C++:\n\nC Code:\n{c_code}\n\nC++ Code:"}
        ]
        all_messages.append(messages)

    # Get TEXT strings (not tokens yet)
    input_texts = []
    for messages in all_messages:
        text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False  # ✅ Get text, not tokens
        )
        input_texts.append(text)

    # Let TOKENIZER handle padding (respects padding_side="left")
    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,  # ✅ This respects padding_side!
        add_special_tokens=False
    ).to(model.device)

    # Terminators
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    # Generate (remove temperature/top_p when do_sample=False)
    outputs = model.generate(
        **inputs,  # ✅ Unpacks input_ids + attention_mask
        max_new_tokens=512,
        eos_token_id=terminators,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=False,  # ✅ No temperature/top_p needed
    )

    # Extract generated tokens
    results = []
    for i, output in enumerate(outputs):
        input_len = inputs['attention_mask'][i].sum().item()  # ✅ Count real tokens
        response = output[input_len:]
        cpp_code = tokenizer.decode(response, skip_special_tokens=True)
        results.append(cpp_code.strip())

    return results

In [11]:
#batching
batch_size = 4
batch_files = []
batch_codes = []
batch_paths = []

for root, _, files in os.walk('input_folder'):
    for file in files:
        if file.endswith('.c'):
            in_path = os.path.join(root, file)
            out_path = in_path.replace('input_folder', 'output_folder').replace('.c', '.cpp')
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            with open(in_path, 'r') as f:
                code = f.read()

            batch_files.append(file)
            batch_codes.append(code)
            batch_paths.append((in_path, out_path))

            # Once batch is full, translate all at once
            if len(batch_codes) == batch_size:
                translations = translate_batch(batch_codes)
                for (in_p, out_p), translation in zip(batch_paths, translations):
                    with open(out_p, 'w') as f_out:
                        f_out.write(translation)
                    print(f"Translated: {in_p} → {out_p}")

                import gc
                gc.collect()
                torch.cuda.empty_cache()
                # Clear batch lists
                batch_files = []
                batch_codes = []
                batch_paths = []




# Translate any remaining files smaller than batch size
if batch_codes:
    translations = translate_batch(batch_codes)
    for (in_p, out_p), translation in zip(batch_paths, translations):
        with open(out_p, 'w') as f_out:
            f_out.write(translation)
        print(f"Translated: {in_p} → {out_p}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Translated: input_folder/C/2709.c → output_folder/C/2709.cpp
Translated: input_folder/C/638.c → output_folder/C/638.cpp
Translated: input_folder/C/9368.c → output_folder/C/9368.cpp
Translated: input_folder/C/9687.c → output_folder/C/9687.cpp
Translated: input_folder/C/13472.c → output_folder/C/13472.cpp
Translated: input_folder/C/2508.c → output_folder/C/2508.cpp
Translated: input_folder/C/12321.c → output_folder/C/12321.cpp
Translated: input_folder/C/7086.c → output_folder/C/7086.cpp
Translated: input_folder/C/2096.c → output_folder/C/2096.cpp
Translated: input_folder/C/2199.c → output_folder/C/2199.cpp
Translated: input_folder/C/10940.c → output_folder/C/10940.cpp
Translated: input_folder/C/2103.c → output_folder/C/2103.cpp
Translated: input_folder/C/9106.c → output_folder/C/9106.cpp
Translated: input_folder/C/4857.c → output_folder/C/4857.cpp
Translated: input_folder/C/1616.c → output_folder/C/1616.cpp
Translated: input_folder/C/1706.c → output_folder/C/1706.cpp
Translated: input_fo

In [12]:
from google.colab import files as colab_files  # CHANGED: Added alias

In [16]:
# 6. Compress and download
print("\nCreating output ZIP...")
!zip -r output.zip output_folder
colab_files.download('output.zip')  # CHANGED: Uses alias
print("Done! Download should start automatically.")


Creating output ZIP...
updating: output_folder/ (stored 0%)
updating: output_folder/C/ (stored 0%)
updating: output_folder/C/1434.cpp (deflated 61%)
updating: output_folder/C/7305.cpp (deflated 61%)
updating: output_folder/C/1541.cpp (deflated 66%)
updating: output_folder/C/2200.cpp (deflated 58%)
updating: output_folder/C/8789.cpp (deflated 59%)
updating: output_folder/C/836.cpp (deflated 57%)
updating: output_folder/C/1432.cpp (deflated 66%)
updating: output_folder/C/2138.cpp (deflated 52%)
updating: output_folder/C/1715.cpp (deflated 62%)
updating: output_folder/C/13801.cpp (deflated 51%)
updating: output_folder/C/13544.cpp (deflated 53%)
updating: output_folder/C/636.cpp (deflated 54%)
updating: output_folder/C/2076.cpp (deflated 58%)
updating: output_folder/C/13443.cpp (deflated 55%)
updating: output_folder/C/9106.cpp (deflated 56%)
updating: output_folder/C/1862.cpp (deflated 57%)
updating: output_folder/C/13539.cpp (deflated 61%)
updating: output_folder/C/9260.cpp (deflated 59%

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Download should start automatically.


In [14]:
print(model.generation_config)


GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "temperature": 0.6,
  "top_p": 0.9
}

